# Luxury Fragrance Dataset — Exploratory Analysis

Initial exploration of the luxury fragrance dataset stored as Parquet.

Objectives:

- Understand the dataset structure
- Identify the fragrance and brand fields
- Determine the number of unique brands
- Examine brand distribution
- Check data quality and missing values
- Identify potential brand-name inconsistencies

In [0]:
# Load the original marketplace dataset once and persist it
# as our project-owned Delta table with a permanent technical ID.

from pyspark.sql import functions as F

source_path = (
    "/Volumes/aidc_inc_luxury_perfume_prices_full/"
    "aidcdatasetshare/full_free_2025_luxury_perfume_prices"
)

target_schema = "main.fragrance_analytics"
target_table = f"{target_schema}.marketplace_base"

# Create the project schema if it does not exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

# Check whether the project table already exists.
if spark.catalog.tableExists(target_table):

    # Table already exists → load the existing IDs.
    # DO NOT generate new IDs.
    df = spark.table(target_table)

else:

    # First run → load the original shared dataset.
    df = spark.read.parquet(source_path)

    # Generate the technical ID ONCE.
    df = df.withColumn(
        "id",
        F.monotonically_increasing_id()
    )

    # Persist the dataset with the permanent IDs.
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    # Reload the persisted table.
    df = spark.table(target_table)

print(f"Table: {target_table}")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

df.printSchema()

display(df.limit(20))

In [0]:
# Dataset overview
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

df.printSchema()

# Inspect columns and sample rows
print(df.columns)
display(df.limit(20))

# Check missing values
missing_values = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(missing_values)

# Explore unique brands
brand_list = (
    df.select("brand")
      .where(F.col("brand").isNotNull())
      .distinct()
      .orderBy("brand")
)

display(brand_list)

# Count unique brands
print(f"Unique brands: {brand_list.count():,}")

# Count fragrances by brand
brand_counts = (
    df.groupBy("brand")
      .count()
      .orderBy(F.col("count").desc())
)

display(brand_counts)

# Explore fragrance types
display(
    df.groupBy("type")
      .count()
      .orderBy(F.col("count").desc())
)

display(
    df.groupBy("perfumeType")
      .count()
      .orderBy(F.col("count").desc())
)

# Explore brand and fragrance details
display(
    df.select(
        "id",
        "brand",
        "title",
        "type",
        "perfumeType",
        "price",
        "priceWithCurrency"
    )
    .orderBy("brand", "title")
    .limit(100)
)